# Estimate noise correlations along the x, y, t, T axes before and after pre-denosing

## Define mask

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath("../src"))
from denoising.eval.empirical_noise_correlations import *
from denoising.figures.empirical_noise_correlations_plots import *
from denoising.data.data_utils import *


# Configuration cell

In [ ]:
# this is the only cell where you need to adapt parameters

subject_ids = [f"P0{i}" for i in range(3, 9)]

#subject_ids = [f"Simulated_Lesion_double_{i}" for i in range(1, 7)]

#subject_ids = ["output_4"]

methods = {
    "No denoising": {"type": "raw"},
    "tMPPCA": {"type": "precomputed", "method": "tMPPCA_5D"},
    "Spin SVD": {"type": "callable", "fn": apply_low_rank_5d_to_dataset_list, "kwargs": {"rank": 8}},
}

extra_axes = ["t", "T"]# ["t", "T"]   # oder nur ["t"]

#configure spatial noise mask:
mask_source = "method"  # "raw" or "method": raw from the noisy data, method from the method directly
mask_type = "fid_window" # fid_window, fid or spatial possible
fid_window = (0.4, 0.9)

# no effect for mask type fid_window
percentile = 10

## datasets
suffix = "normalized" #"normalized" #"normalized_uncorrelated_noise"
base_dir = "../datasets/" #"../datasets"

In [ ]:
results = run_noise_analysis_pipeline(
    subject_ids=subject_ids,
    methods=methods,
    extra_axes=extra_axes,
    mask_source=mask_source,
    mask_type=mask_type,
    fid_window=fid_window,
    percentile=percentile,
    suffix=suffix,
    base_dir=base_dir,
)

In [ ]:
# check that noise mask makes sense: (only applicaple to fid_window)

plot_mean_fid_with_noise_mask(
    results,
    methods=["No denoising"],
    title="Mean FID and selected noise region"
)

In [ ]:
plot_spatial_acf_comparison(
    results["spatial_stats_by_method"],
    axes_to_plot=(0, 1, 2),
    max_lag_plot=22,
    title="Spatial noise correlations"
)

In [ ]:
plot_acf_comparison(
    results["acf_stats_by_method"],
    axis_idx=results["axis_indices"],
    axis_names=results["extra_axes"],
    title="Noise autocorrelation"
)